# Bat / Non-bat Acoustic Classification — Training Pipeline

This notebook trains a Random Forest classifier to distinguish recordings containing bat vocalizations from non-bat recordings. The classifier is intended as a post-processing step to reduce false-positive detections generated by autonomous ultrasonic recorders and automatic species-identification software, thereby decreasing the amount of manual review required.

The workflow was developed as a proof of concept for a course project. It includes dataset validation, minimal audio preprocessing, acoustic feature extraction, model training, model evaluation, and export of the trained model for use in a separate prediction notebook.


The metadata file must contain at least:

- `filename`: exact audio filename, including the extension;
- `classification`: either `bat` or `non-bat`.



## 1. Connect Google Drive

This step mounts Google Drive so that the notebook can access the metadata, audio files, generated outputs, and saved model.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Project configuration

All input and output paths are defined in one place. Change `PROJECT_DIR` only if the project folder is stored elsewhere in Google Drive.


In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/bat_project')
OUTPUT_DIR = PROJECT_DIR / "outputs"
CSV_PATH = PROJECT_DIR / 'bat_audio_metadata_annotated.csv'
AUDIO_DIR = PROJECT_DIR / 'audio'

OUTPUT_DIR.mkdir(exist_ok=True)

print('Project directory:', PROJECT_DIR)
print('CSV file exists:', CSV_PATH.exists())
print('Audio directory exists:', AUDIO_DIR.exists())

## 3. Record the computational environment



In [ ]:
import sys
import numpy as np
import pandas as pd
import sklearn

print("Python version:", sys.version)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("Scikit-learn version:", sklearn.__version__)


## 4. Load and validate metadata

The metadata table links each audio file to its known class label. Column names are stripped of accidental leading or trailing spaces before analysis.


In [ ]:
import pandas as pd

metadata = pd.read_csv(CSV_PATH)

metadata.columns = metadata.columns.str.strip()

display(metadata.head())

print('Column names:')
print(metadata.columns.tolist())

print('\nClass distribution:')
print(metadata['classification'].value_counts())

### File matching check

This check confirms that every filename listed in the CSV exists in the audio directory. A result of zero missing files is required before continuing.


In [ ]:
missing_files = []

for filename in metadata['filename'].astype(str).str.strip():
    audio_path = AUDIO_DIR / filename

    if not audio_path.exists():
        missing_files.append(filename)

print('Total records in CSV:', len(metadata))
print('Missing audio files:', len(missing_files))

missing_files[:20]

## 5. Install and import audio-processing libraries

`librosa` is used for loading, resampling, visualizing, and extracting acoustic features. `soundfile` provides audio-file support, and `matplotlib` is used for plots.


In [ ]:
!pip install librosa soundfile matplotlib -q

## 6. Inspect one example recording

A single recording is loaded at its original sample rate for an initial visual check. The waveform and spectrogram help confirm that the file is readable and contains an acoustic signal.


In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt

audio_file = AUDIO_DIR / metadata.iloc[0]["filename"]

signal, sr = librosa.load(audio_file, sr=None)

print(f"File: {audio_file.name}")
print(f"Sample rate: {sr} Hz")
print(f"Duration: {len(signal)/sr:.2f} s")

In [ ]:
#Waveform plot
plt.figure(figsize=(12,4))

librosa.display.waveshow(signal, sr=sr)

plt.title("Waveform")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")

plt.show()

In [ ]:
#Spectogram plot
import numpy as np

S = librosa.amplitude_to_db(
    np.abs(librosa.stft(signal)),
    ref=np.max
)

plt.figure(figsize=(12,5))

librosa.display.specshow(
    S,
    sr=sr,
    x_axis="time",
    y_axis="hz"
)

plt.colorbar(label="dB")
plt.title("Spectrogram")

plt.show()

## 7. Minimal audio preprocessing

Most recordings had already been detected, filtered, or segmented before inclusion in the dataset. Therefore, preprocessing is intentionally conservative.

For each recording, the function:

1. loads the signal and converts it to mono;
2. resamples it to 256 kHz so that feature extraction uses a common time and frequency scale;
3. replaces invalid numerical values;
4. removes the DC offset;
5. applies peak normalization.

No additional noise reduction, frequency filtering, or silence trimming is applied.

**Important limitation:** resampling a low-sample-rate file to 256 kHz does not recreate ultrasonic frequencies that were absent from the original recording.


In [ ]:
def preprocess_audio(
    audio_path,
    normalize=True
):
    """
    Load one audio file using minimal preprocessing.

    Parameters
    ----------
    audio_path : str or pathlib.Path
        Path to the audio file.

    normalize : bool
        Whether to apply peak-amplitude normalization.
    """

    TARGET_SAMPLE_RATE = 256000

    # Load audio at its original sampling rate
    signal, original_sr = librosa.load(
        audio_path,
        sr=None,
        mono=True
    )

    # Resample to the target sampling rate
    if original_sr != TARGET_SAMPLE_RATE:
        signal = librosa.resample(
            signal,
            orig_sr=original_sr,
            target_sr=TARGET_SAMPLE_RATE
        )

    sample_rate = TARGET_SAMPLE_RATE

    original_samples = len(signal)

    # Replace NaN and infinite values
    signal = np.nan_to_num(
        signal,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    # Remove DC offset
    if len(signal) > 0:
        signal = signal - np.mean(signal)

    # Peak amplitude before normalization
    peak_amplitude = (
        np.max(np.abs(signal))
        if len(signal) > 0
        else 0.0
    )

    # Peak normalization
    if normalize and peak_amplitude > 0:
        signal = signal / peak_amplitude

    processing_info = {
        "original_samples": original_samples,
        "processed_samples": len(signal),
        "duration_seconds": len(signal) / sample_rate,
        "original_sample_rate": original_sr,
        "sample_rate": sample_rate,
        "original_peak_amplitude": float(peak_amplitude)
    }

    return signal, sample_rate, processing_info

### Preprocessing examples

One bat and one non-bat recording are processed to verify that the function returns valid signals, durations, amplitudes, and sample rates.


In [ ]:
#Test preprocessing on one bat and one non-bat recording

bat_example = metadata.loc[
    metadata["classification"] == "bat"
].iloc[0]

non_bat_example = metadata.loc[
    metadata["classification"] == "non-bat"
].iloc[0]

bat_path = AUDIO_DIR / bat_example["filename"]
non_bat_path = AUDIO_DIR / non_bat_example["filename"]

bat_signal, bat_sr, bat_info = preprocess_audio(bat_path)
non_bat_signal, non_bat_sr, non_bat_info = preprocess_audio(non_bat_path)

print("Bat example")
print("File:", bat_path.name)
print("Sample rate:", bat_sr, "Hz")
print("Duration:", round(bat_info["duration_seconds"], 4), "seconds")
print("Peak amplitude before normalization:",
      round(bat_info["original_peak_amplitude"], 6))

print("\nNon-bat example")
print("File:", non_bat_path.name)
print("Sample rate:", non_bat_sr, "Hz")
print("Duration:", round(non_bat_info["duration_seconds"], 4), "seconds")
print("Peak amplitude before normalization:",
      round(non_bat_info["original_peak_amplitude"], 6))

### Preprocessing quality control

The same preprocessing procedure is applied to every recording. The quality-control table records processing status, duration, amplitude, and whether a recording is empty or silent.


In [ ]:
from tqdm.auto import tqdm

preprocessing_records = []
preprocessing_errors = []

for _, row in tqdm(
    metadata.iterrows(),
    total=len(metadata),
    desc="Checking recordings"
):
    audio_path = AUDIO_DIR / row["filename"]

    try:
        signal, sample_rate, info = preprocess_audio(
            audio_path
        )

        preprocessing_records.append({
            "filename": row["filename"],
            "classification": row["classification"],
            "sample_rate": sample_rate,
            "duration_seconds": info["duration_seconds"],
            "number_of_samples": info["processed_samples"],
            "original_peak_amplitude":
                info["original_peak_amplitude"],
            "processed_peak_amplitude": (
                float(np.max(np.abs(signal)))
                if len(signal) > 0
                else 0.0
            ),
            "is_empty": len(signal) == 0,
            "is_silent": (
                np.max(np.abs(signal)) == 0
                if len(signal) > 0
                else True
            )
        })

    except Exception as error:
        preprocessing_errors.append({
            "filename": row["filename"],
            "classification": row["classification"],
            "error": str(error)
        })

preprocessing_qc = pd.DataFrame(preprocessing_records)
preprocessing_errors = pd.DataFrame(preprocessing_errors)

In [ ]:
#Check recordings after processing
print("Successfully processed recordings:",
      len(preprocessing_qc))

print("Processing errors:",
      len(preprocessing_errors))

print("Empty recordings:",
      preprocessing_qc["is_empty"].sum())

print("Silent recordings:",
      preprocessing_qc["is_silent"].sum())

In [ ]:
#Prints the number of recordings under the sample rate
print("Sample-rate distribution:")

display(
    preprocessing_qc["sample_rate"]
    .value_counts()
    .sort_index()
    .rename_axis("sample_rate")
    .reset_index(name="number_of_recordings")
)

In [ ]:
#Prints the sample rate by class (bat/non-bat)
sample_rate_by_class = pd.crosstab(
    preprocessing_qc["sample_rate"],
    preprocessing_qc["classification"]
)

display(sample_rate_by_class)

## 8. Acoustic feature extraction

Each recording is converted into a fixed-length numerical vector suitable for Random Forest classification.

The feature set contains:

- 20 Mel-frequency cepstral coefficients (MFCCs);
- spectral centroid;
- spectral bandwidth;
- spectral rolloff;
- spectral flatness;
- root-mean-square energy;
- zero-crossing rate.

For every feature, the mean and standard deviation across time frames are calculated. This produces 52 predictors per recording.

The MFCC computation uses 40 Mel filters, an FFT window of 2,048 samples, and a hop length of 512 samples.


### Feature extraction

In [ ]:
"""
Feature set

Cepstral features
- MFCCs

Spectral features
- Spectral centroid
- Spectral bandwidth
- Spectral rolloff
- Spectral flatness

Energy
- RMS energy

Temporal
- Zero-crossing rate

For each feature, the mean and standard deviation are calculated.
"""


In [ ]:
   #Extract acoustic features from one recording.

   def extract_features(signal, sample_rate):

    """
    Parameters
    ----------
    signal : numpy.ndarray
        Preprocessed audio signal.

    sample_rate : int
        Sampling rate.

    Returns
    -------
    dict
        Dictionary containing acoustic features.
    """

    features = {}

    #MFCC
    mfcc = librosa.feature.mfcc(
    y=signal,
    sr=sample_rate,
    n_mfcc=20,
    n_mels=40,
    n_fft=2048,
    hop_length=512
)

    for i in range(20):

        features[f"mfcc_{i+1}_mean"] = np.mean(mfcc[i])

        features[f"mfcc_{i+1}_std"] = np.std(mfcc[i])

    #Spectral centroid
    centroid = librosa.feature.spectral_centroid(
        y=signal,
        sr=sample_rate
    )

    features["spectral_centroid_mean"] = np.mean(centroid)
    features["spectral_centroid_std"] = np.std(centroid)

    #Spectral bandwidth
    bandwidth = librosa.feature.spectral_bandwidth(
        y=signal,
        sr=sample_rate
    )

    features["spectral_bandwidth_mean"] = np.mean(bandwidth)
    features["spectral_bandwidth_std"] = np.std(bandwidth)

    #Spectral rolloff
    rolloff = librosa.feature.spectral_rolloff(
        y=signal,
        sr=sample_rate
    )

    features["spectral_rolloff_mean"] = np.mean(rolloff)
    features["spectral_rolloff_std"] = np.std(rolloff)

    #Spectral flatness
    flatness = librosa.feature.spectral_flatness(
        y=signal
    )

    features["spectral_flatness_mean"] = np.mean(flatness)
    features["spectral_flatness_std"] = np.std(flatness)

    #RMS Energy
    rms = librosa.feature.rms(
        y=signal
    )

    features["rms_mean"] = np.mean(rms)
    features["rms_std"] = np.std(rms)

    #Zero-crossing rate
    zcr = librosa.feature.zero_crossing_rate(
        signal
    )

    features["zcr_mean"] = np.mean(zcr)
    features["zcr_std"] = np.std(zcr)

    #Return
    return features

In [ ]:
#test if all features were extracted correctly
test_features = extract_features(
    bat_signal,
    bat_sr
)

print(f"Number of extracted features: {len(test_features)}")

print(
    "All features are finite:",
    all(np.isfinite(list(test_features.values())))
)

In [ ]:
#visualize features and respective values
import pandas as pd

feature_table = (
    pd.DataFrame(test_features, index=[0])
    .T
    .rename(columns={0: "value"})
)

display(feature_table)

## 9. Build the feature dataset

The preprocessing and feature-extraction functions are applied to all recordings. Each row of the resulting table represents one audio file, and each predictor column represents a summarized acoustic feature.

The filename and class label are retained for traceability but are excluded from model predictors.


## Building the feature dataset

In [ ]:
#Extract features from all recordings
from tqdm.auto import tqdm

feature_dataset = []
feature_errors = []

for _, row in tqdm(
    metadata.iterrows(),
    total=len(metadata),
    desc="Extracting acoustic features"
):

    audio_path = AUDIO_DIR / row["filename"]

    try:

        signal, sample_rate, processing_info = preprocess_audio(audio_path)

        features = extract_features(signal, sample_rate)

        features["filename"] = row["filename"]
        features["classification"] = row["classification"]

        # Metadata used for grouped validation
        features["recording_group"] = row["recording_group"]
        features["recorder"] = row["recorder"]
        features["date"] = row["date"]
        features["source"] = row["source"]

        feature_dataset.append(features)

    except Exception as error:

        feature_errors.append({
            "filename": row["filename"],
            "classification": row["classification"],
            "error": str(error)
        })

In [ ]:
#Create the feature table

feature_dataset = pd.DataFrame(feature_dataset)
feature_errors = pd.DataFrame(feature_errors)

print("Successfully processed recordings:",
      len(feature_dataset))

print("Errors:",
      len(feature_errors))

In [ ]:
#Inspect the dataset
display(feature_dataset.head())
print(feature_dataset.shape)

In [ ]:
#Save the dataset
feature_dataset.to_csv(
    OUTPUT_DIR / "feature_dataset.csv",
    index=False
)

print("Saved:",
      OUTPUT_DIR / "feature_dataset.csv")

## 10. Train–test split

The dataset is divided into:

- 80% training data;
- 20% testing data.

Stratification preserves the proportion of bat and non-bat recordings in both subsets. `random_state=42` makes the split reproducible.


## Train-test split

In [ ]:
from sklearn.model_selection import train_test_split

metadata_columns = [
    "filename",
    "classification",
    "recording_group",
    "recorder",
    "date",
    "source"
]

#Predictor matrix: acoustic features only.
X = feature_dataset.drop(columns=metadata_columns)

#Binary response variable:
#bat = recording contains bat acoustic activity
#non-bat = no bat acoustic activity detected in the labelled segment
y = feature_dataset["classification"]

groups = feature_dataset["recording_group"]

print("Feature matrix:", X.shape)
print("Number of recording groups:", groups.nunique())
print("\nClass distribution:")
print(y.value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

## 11. Random Forest training

The model uses 200 decision trees. Balanced class weights reduce the influence of small differences in class size. The fixed random seed makes model fitting reproducible under the same software environment and dataset.


## Model Training

In [ ]:
from sklearn.ensemble import RandomForestClassifier

#Random Forest hyperparameters used for the final classifier.
#Keep these values unchanged when reproducing the reported results.
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
#class_weight="balanced" adjusts class weights according to class frequency.
#This is used instead of manually duplicating or removing recordings.
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

print("Random Forest training completed.")

In [ ]:
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)

## 12. Model evaluation

Performance is first evaluated on the held-out test set using:

- precision;
- recall;
- F1-score;
- accuracy;
- confusion matrix.

For this application, recall for the `bat` class is especially relevant because it measures how many true bat recordings are retained by the classifier.


## Evaluate model performance

In [ ]:
#Classification report
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
#Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=rf_model.classes_
)

fig, ax = plt.subplots(figsize=(5,5))

disp.plot(ax=ax, cmap="Blues", colorbar=False)

plt.title("Confusion Matrix")
plt.show()

In [ ]:
#Accuracy
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")

In [ ]:
#Feature importance
importance = (
    pd.DataFrame({
        "Feature": X.columns,
        "Importance": rf_model.feature_importances_
    })
    .sort_values("Importance", ascending=False)
)

display(importance.head(10))

### Grouped cross-validation

Five-fold Stratified Group K-Fold cross-validation is used as the main
model evaluation.

Recordings are grouped by `recording_group` (recorder + recording date).

Stratification attempts to maintain class balance across folds, although
perfect balance cannot be guaranteed because recording groups vary
substantially in size and class composition.

In [ ]:
#Grouped validation keeps recordings from the same recording_group
#together in the same fold to reduce leakage from acoustically related files.
from sklearn.model_selection import StratifiedGroupKFold, cross_validate

group_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1": "f1_macro"
}

rf_group_results = cross_validate(
    rf_model,
    X,
    y,
    groups=groups,
    cv=group_cv,
    scoring=scoring
)

for metric in scoring:

    scores = rf_group_results[f"test_{metric}"]

    print(
        f"{metric}: "
        f"{scores.mean():.3f} ± {scores.std():.3f}"
    )

    print("Fold scores:", scores)
    print()

In [ ]:
fold_summary = []

for fold, (train_idx, test_idx) in enumerate(
    group_cv.split(X, y, groups),
    start=1
):

    test_metadata = feature_dataset.iloc[test_idx]

    fold_summary.append({
        "fold": fold,
        "n_recordings": len(test_idx),
        "n_groups": test_metadata["recording_group"].nunique(),
        "bat": (
            test_metadata["classification"] == "bat"
        ).sum(),
        "non_bat": (
            test_metadata["classification"] == "non-bat"
        ).sum()
    })

fold_summary = pd.DataFrame(fold_summary)

display(fold_summary)

### Grouped validation predictions

Predictions from each validation fold were stored to examine
misclassified recordings and variation in model performance among
recording groups.

In [ ]:
from sklearn.base import clone

grouped_predictions = []

for fold, (train_idx, test_idx) in enumerate(
    group_cv.split(X, y, groups),
    start=1
):

    model = clone(rf_model)

    X_train_fold = X.iloc[train_idx]
    X_test_fold = X.iloc[test_idx]

    y_train_fold = y.iloc[train_idx]
    y_test_fold = y.iloc[test_idx]

    model.fit(
        X_train_fold,
        y_train_fold
    )

    predicted = model.predict(
        X_test_fold
    )

    bat_index = list(
        model.classes_
    ).index("bat")

    probability_bat = model.predict_proba(
        X_test_fold
    )[:, bat_index]

    fold_results = feature_dataset.iloc[
        test_idx
    ][
        [
            "filename",
            "recording_group",
            "recorder",
            "source"
        ]
    ].copy()

    fold_results["fold"] = fold
    fold_results["true_class"] = (
        y_test_fold.values
    )
    fold_results["predicted_class"] = (
        predicted
    )
    fold_results["probability_bat"] = (
        probability_bat
    )

    fold_results["correct"] = (
        fold_results["true_class"]
        ==
        fold_results["predicted_class"]
    )

    grouped_predictions.append(
        fold_results
    )

grouped_predictions = pd.concat(
    grouped_predictions,
    ignore_index=True
)

In [ ]:
fold4 = grouped_predictions[
    grouped_predictions["fold"] == 4
]

display(
    pd.crosstab(
        fold4["true_class"],
        fold4["predicted_class"]
    )
)

In [ ]:
display(
    fold4.groupby(
        [
            "recording_group",
            "true_class"
        ]
    ).agg(
        n=("filename", "count"),
        accuracy=("correct", "mean"),
        mean_probability_bat=(
            "probability_bat",
            "mean"
        )
    ).sort_values(
        "accuracy"
    )
)

In [ ]:
grouped_predictions.to_csv(
    OUTPUT_DIR /
    "grouped_validation_predictions.csv",
    index=False
)

## 13. Confidence-based screening

Because the classifier is intended as a screening tool, prediction probabilities are used to separate high-confidence classifications from uncertain recordings requiring expert review.

The following thresholds are used:

- **P(bat) ≤ 0.30:** NON-BAT
- **0.30 < P(bat) < 0.70:** EXPERT REVIEW
- **P(bat) ≥ 0.70:** BAT

These thresholds prioritize retention of bat recordings while reducing manual review. False-positive BAT classifications are considered less problematic than false-negative NON-BAT classifications because BAT predictions remain available for subsequent inspection, whereas recordings automatically classified as NON-BAT may be excluded.

Threshold performance is evaluated on the held-out test set. Grouped cross-validation is retained separately as a stricter assessment of generalization across recording groups.


In [ ]:
# Build a table containing held-out test predictions and probabilities

test_results = feature_dataset.loc[
    X_test.index,
    [
        "filename",
        "recording_group",
        "recorder",
        "source",
        "classification"
    ]
].copy()

test_results["true_class"] = y_test.values
test_results["predicted_class"] = y_pred

bat_index = list(rf_model.classes_).index("bat")
test_results["probability_bat"] = y_prob[:, bat_index]


def assign_review_status(probability_bat):

    if probability_bat >= 0.70:
        return "BAT"

    elif probability_bat <= 0.30:
        return "NON-BAT"

    else:
        return "EXPERT REVIEW"


test_results["review_status"] = (
    test_results["probability_bat"]
    .apply(assign_review_status)
)


In [ ]:
# Evaluate the screening policy on the held-out test set

screening_test = pd.crosstab(
    test_results["true_class"],
    test_results["review_status"]
)

display(screening_test)

n_true_bat = (test_results["true_class"] == "bat").sum()

n_bat_retained = (
    (test_results["true_class"] == "bat")
    &
    (test_results["review_status"].isin(["BAT", "EXPERT REVIEW"]))
).sum()

bat_retention = n_bat_retained / n_true_bat

print(f"\nBat retention rate: {bat_retention:.3f} ({bat_retention:.1%})")
print(f"Bat recordings retained: {n_bat_retained}/{n_true_bat}")


### Screening interpretation

With the selected thresholds, recordings predicted with intermediate confidence are retained for expert review rather than being automatically discarded.


In [ ]:
# Estimate review workload across grouped out-of-fold predictions

grouped_predictions["review_status"] = (
    grouped_predictions["probability_bat"]
    .apply(assign_review_status)
)

review_summary = (
    grouped_predictions["review_status"]
    .value_counts()
    .rename_axis("review_status")
    .reset_index(name="n")
)

review_summary["percentage"] = (
    review_summary["n"]
    / len(grouped_predictions)
    * 100
)

display(review_summary)

automatic_n = (
    grouped_predictions["review_status"] != "EXPERT REVIEW"
).sum()

review_n = (
    grouped_predictions["review_status"] == "EXPERT REVIEW"
).sum()

print(
    f"\nAutomatically assigned: {automatic_n}/{len(grouped_predictions)} "
    f"({automatic_n / len(grouped_predictions):.1%})"
)

print(
    f"Sent to expert review: {review_n}/{len(grouped_predictions)} "
    f"({review_n / len(grouped_predictions):.1%})"
)


### Estimated manual-review workload

Using the 0.30/0.70 thresholds, the grouped out-of-fold predictions provide an estimate of how much of the dataset would be automatically assigned versus sent to expert review.

These grouped out-of-fold predictions are used here to estimate workload, not to select the final thresholds. Their lower classification performance also highlights limited generalization to some unseen recording groups.


##Leave-One-Recorder-Out validation


In [ ]:
# Inspect recordings available for each recorder

recorder_summary = (
    feature_dataset
    .groupby("recorder")
    .agg(
        n_recordings=("filename", "count"),
        n_bat=("classification", lambda x: (x == "bat").sum()),
        n_non_bat=("classification", lambda x: (x == "non-bat").sum())
    )
    .sort_values("n_recordings", ascending=False)
)

display(recorder_summary)

In [ ]:
# Autonomous recorders included in Leave-One-Recorder-Out validation

loro_recorders = [
    "S4U15525",
    "TREVO88",
    "SMU00657",
    "PETER69"
]

print("Recorders included in LORO:")
print(loro_recorders)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np

# Recorder information aligned exactly with X
recorder_labels = feature_dataset.loc[X.index, "recorder"]

recorders = recorder_labels.dropna().unique()

loro_predictions = []

for held_out_recorder in recorders:

    print(f"Testing recorder: {held_out_recorder}")

    # Masks aligned with X
    test_mask = recorder_labels == held_out_recorder
    train_mask = recorder_labels != held_out_recorder

    X_train_loro = X.loc[train_mask]
    X_test_loro = X.loc[test_mask]

    y_train_loro = y.loc[train_mask]
    y_test_loro = y.loc[test_mask]

    # Safety check
    if len(X_test_loro) == 0:
        print(f"  Skipping {held_out_recorder}: no test samples.")
        continue

    if len(X_train_loro) == 0:
        print(f"  Skipping {held_out_recorder}: no training samples.")
        continue

    model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )

    model.fit(
        X_train_loro,
        y_train_loro
    )

    predicted = model.predict(
        X_test_loro
    )

    probability_bat = model.predict_proba(
        X_test_loro
    )[:, list(model.classes_).index("bat")]

    predictions = pd.DataFrame({
        "filename": feature_dataset.loc[X_test_loro.index, "filename"],
        "recorder": held_out_recorder,
        "true_class": y_test_loro,
        "predicted_class": predicted,
        "probability_bat": probability_bat
    })

    loro_predictions.append(predictions)

# Combine all valid recorder folds
loro_predictions = pd.concat(
    loro_predictions,
    ignore_index=True
)

display(loro_predictions.head())

In [ ]:
loro_results = pd.DataFrame(loro_predictions)

display(loro_results)

In [ ]:
print(type(loro_predictions))
print(loro_predictions.shape)

display(loro_predictions.head())

In [ ]:
for recorder in loro_recorders:

    subset = loro_predictions[
        loro_predictions["recorder"] == recorder
    ]

    print(f"\nRecorder: {recorder}")

    display(
        pd.crosstab(
            subset["true_class"],
            subset["predicted_class"]
        )
    )

    print(
        "Mean P(bat):",
        subset["probability_bat"].mean()
    )

Leave-One-Recorder-Out validation provides a stricter assessment of generalization to recordings from a recorder that is completely absent from model training.

Performance varies among recorders, indicating that recorder-specific acoustic conditions and unequal class representation can affect generalization. Because the available dataset does not contain equally balanced bat and non-bat examples for every recorder, recorder effects cannot be separated cleanly from differences in class composition and recording conditions.

These results are therefore treated as a limitation and as motivation for collecting more balanced bat and non-bat examples across recorder types.


## 14. Save the trained model bundle

The exported bundle contains:

- the trained Random Forest model;
- the exact predictor-column order;
- the target sample rate.

Keeping these items together ensures that the separate prediction notebook applies new recordings in the same format used during training.


In [ ]:
# Train final Random Forest using the complete dataset

final_rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

final_rf_model.fit(X, y)

In [ ]:
#Save all information required to reproduce predictions.
#The prediction notebook should read these values from the bundle rather
#than redefining them manually.
model_bundle = {
    "model": final_rf_model,
    "feature_columns": list(X.columns),
    "target_sample_rate": 256000,
    "non_bat_threshold": 0.30,
    "bat_threshold": 0.70
}


In [ ]:
import joblib

joblib.dump(
    model_bundle,
    OUTPUT_DIR / "bat_classifier_model.joblib"
)

print(
    "Saved:",
    OUTPUT_DIR / "bat_classifier_model.joblib"
)

## Reproducibility notes and limitations

### Reproducibility

This workflow was designed to provide a reproducible proof-of-concept for binary acoustic screening of bat and non-bat recordings.

To reproduce the analysis:

- use the same labelled audio dataset and metadata file;
- keep the original filenames, class labels, and recording-group information;
- run all notebook cells sequentially in a fresh Google Colab session;
- use the same audio preprocessing and acoustic feature-extraction functions in both the training and prediction notebooks;
- retain the fixed random seed (`random_state=42`) for model fitting and validation procedures;
- record the Python and package versions used for the analysis.

All recordings are resampled to 256 kHz before feature extraction to standardize the input data. The final Random Forest model is trained using the complete labelled dataset after model evaluation and selection of the screening policy.

### Validation strategy

A random 80/20 train-test split is used as the baseline evaluation and as the basis for evaluating the final confidence-based screening policy.

Recordings collected during the same recorder-night may share acoustic or recording characteristics, potentially inflating performance when related recordings occur in both training and testing subsets. For this reason, Stratified Group K-Fold cross-validation is also included as a stricter assessment of generalization across recording groups.

Leave-One-Recorder-Out validation provides an additional stress test for generalization to unseen recorders. These grouped validation analyses are interpreted as limitations and complementary generalization tests rather than as the basis for selecting the screening thresholds.

### Limitations

The available dataset does not provide a fully balanced representation of bat and non-bat sounds across recorders, recording conditions, locations, and sound sources. Some recorder/class combinations are much better represented than others.

Grouped and recorder-level validation showed substantially lower performance than the random held-out split for some groups, indicating limited generalization to acoustic conditions that are poorly represented during training.

Random Forest probability estimates are used as practical confidence scores for screening and should not be interpreted as formally calibrated probabilities of biological certainty.

Additional labelled recordings from multiple recorders and recording environments, ideally containing both bat and non-bat examples for each recording system, would improve future evaluation and model generalization.

### Intended use

The exported model is intended as a screening tool for candidate recordings produced by autonomous acoustic recorders or automatic detection software.

The final workflow uses the following confidence thresholds:

- `P(bat) ≤ 0.30` → **NON-BAT**
- `0.30 < P(bat) < 0.70` → **EXPERT REVIEW**
- `P(bat) ≥ 0.70` → **BAT**

On the held-out test set, this screening policy retained 63 of 64 true bat recordings as either BAT or EXPERT REVIEW, corresponding to a **98.4% bat retention rate**. Only one true bat recording was automatically classified as NON-BAT.

Across grouped out-of-fold predictions for the complete dataset, the same thresholds assigned approximately **76% of recordings automatically** and sent approximately **24% to expert review**, substantially reducing the manual-review workload compared with the original 0.10/0.90 screening rule.

Recordings assigned to **EXPERT REVIEW** should be manually inspected by an experienced analyst. The tool is intended to reduce the volume of recordings requiring manual review rather than replace expert acoustic identification.
